# Homework 7.1 - Part IV

## Build a Small Conditioned Dataset

This notebook constructs a toy conditioned-generation dataset of paired records

$$
\boxed{
c
\;\text{paired with}\;
\Gamma_c
\;\text{paired with}\;
I_c.
}
$$

The assignment sometimes writes this dataset association with double arrows. Here the notation means that the three objects are stored together as corresponding records; it does **not** mean the maps are invertible. Part III showed that the rendering map $R:\Gamma_c\mapsto I_c$ is many-to-one.

For every sample we store:

1. the condition vector

$$
c
=
(\sigma,\rho,\beta,x_0,y_0,z_0,T);
$$

2. the numerical trajectory

$$
\Gamma_c;
$$

3. the rendered trajectory image

$$
I_c.
$$

The trajectory is the mathematically meaningful object.

The image is a deterministic visual representation of that trajectory.

# 1. Dataset design

The design is fixed before simulation.

We generate

$$
N=120
$$

independent conditions.

The split is

- 84 training conditions;
- 18 validation conditions;
- 18 test conditions.

The random seed is

$$
63871.
$$

Parameters are sampled independently and uniformly from

$$
\sigma\in[8,12],
$$

$$
\rho\in[24,32],
$$

$$
\beta\in[2.2,3.0],
$$

$$
x_0,y_0\in[-15,15],
$$

$$
z_0\in[5,35].
$$

The integration horizon is fixed at

$$
T=25
$$

with

$$
\Delta t=0.01.
$$

Thus every stored trajectory has exactly

$$
2501
$$

time samples.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from PIL import Image, ImageDraw, ImageOps
import matplotlib.pyplot as plt

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "hw7_1"
RESULT_DIR = PROJECT_ROOT / "result images" / "hw7_1"

DATASET_DIR = (
    DATA_DIR
    / "part4_conditioned_dataset"
)

TRAJECTORY_DIR = (
    DATASET_DIR
    / "trajectories"
)

IMAGE_DIR = (
    DATASET_DIR
    / "images"
)

for path in [
    DATASET_DIR,
    TRAJECTORY_DIR,
    IMAGE_DIR,
    RESULT_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

part1_config = json.loads(
    (
        DATA_DIR
        / "part1_pipeline_configuration.json"
    ).read_text(
        encoding="utf-8"
    )
)

ranges = (
    part1_config[
        "dataset_parameter_domain"
    ]
)

SEED = 63871

N_TRAIN = 84
N_VALIDATION = 18
N_TEST = 18
N_TOTAL = (
    N_TRAIN
    + N_VALIDATION
    + N_TEST
)

T = 25.0
DT = 0.01

H = 256
W = 256
LINE_WIDTH = 1

rng = np.random.default_rng(
    SEED
)

print("Dataset size:", N_TOTAL)
print(
    "Split:",
    N_TRAIN,
    N_VALIDATION,
    N_TEST,
)

Dataset size: 120
Split: 84 18 18


# 2. Sample the condition vectors

In [2]:
def sample_uniform(
    low,
    high,
    size,
):
    return rng.uniform(
        low,
        high,
        size=size,
    )


conditions = pd.DataFrame({
    "sample_id": np.arange(
        N_TOTAL,
        dtype=int,
    ),
    "sigma": sample_uniform(
        *ranges["sigma"],
        N_TOTAL,
    ),
    "rho": sample_uniform(
        *ranges["rho"],
        N_TOTAL,
    ),
    "beta": sample_uniform(
        *ranges["beta"],
        N_TOTAL,
    ),
    "x0": sample_uniform(
        *ranges["x0"],
        N_TOTAL,
    ),
    "y0": sample_uniform(
        *ranges["y0"],
        N_TOTAL,
    ),
    "z0": sample_uniform(
        *ranges["z0"],
        N_TOTAL,
    ),
    "T": np.full(
        N_TOTAL,
        T,
    ),
})

conditions["split"] = (
    ["train"] * N_TRAIN
    + ["validation"] * N_VALIDATION
    + ["test"] * N_TEST
)

condition_path = (
    DATASET_DIR
    / "conditions.csv"
)

conditions.to_csv(
    condition_path,
    index=False,
)

print(
    conditions.groupby(
        "split"
    ).size()
)

print()
print(
    conditions.head().to_string(
        index=False
    )
)

print()
print("Saved:", condition_path.resolve())

split
test          18
train         84
validation    18
dtype: int64

 sample_id     sigma       rho     beta        x0        y0        z0    T split
         0  8.534315 24.161536 2.379947 13.316783 10.676565 32.206152 25.0 train
         1  9.378573 31.599074 2.469527 -9.781024 -4.023229 13.656476 25.0 train
         2 10.764040 25.792990 2.760216 10.097013 -8.536443 30.055130 25.0 train
         3  9.841594 26.358556 2.944859 12.825211  9.448849  6.131445 25.0 train
         4  8.979528 27.334787 2.751819  1.531599 -9.415211  9.494693 25.0 train

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.1\data\hw7_1\part4_conditioned_dataset\conditions.csv


# 3. Simulate every Lorenz trajectory

Each sample is integrated independently from its own initial condition and parameter vector.

In [3]:
def lorenz_rhs(
    t,
    state,
    sigma,
    rho,
    beta,
):
    x, y, z = state

    return np.array([
        sigma * (y - x),
        x * (rho - z) - y,
        x * y - beta * z,
    ], dtype=float)


time = np.arange(
    0.0,
    T + 0.5 * DT,
    DT,
)

simulation_rows = []

for row in conditions.itertuples():
    initial_state = np.array([
        row.x0,
        row.y0,
        row.z0,
    ], dtype=float)

    solution = solve_ivp(
        fun=lambda t, state: lorenz_rhs(
            t,
            state,
            row.sigma,
            row.rho,
            row.beta,
        ),
        t_span=(0.0, T),
        y0=initial_state,
        t_eval=time,
        method="DOP853",
        rtol=1e-9,
        atol=1e-11,
    )

    if not solution.success:
        raise RuntimeError(
            f"Simulation failed for sample {row.sample_id}: "
            f"{solution.message}"
        )

    states = solution.y.T

    trajectory_path = (
        TRAJECTORY_DIR
        / f"sample_{row.sample_id:03d}.npz"
    )

    np.savez_compressed(
        trajectory_path,
        time=solution.t,
        states=states,
        condition=np.array([
            row.sigma,
            row.rho,
            row.beta,
            row.x0,
            row.y0,
            row.z0,
            row.T,
        ], dtype=float),
    )

    simulation_rows.append({
        "sample_id": int(
            row.sample_id
        ),
        "split": row.split,
        "solver_success": True,
        "samples": len(
            solution.t
        ),
        "x_min": float(
            states[:, 0].min()
        ),
        "x_max": float(
            states[:, 0].max()
        ),
        "y_min": float(
            states[:, 1].min()
        ),
        "y_max": float(
            states[:, 1].max()
        ),
        "z_min": float(
            states[:, 2].min()
        ),
        "z_max": float(
            states[:, 2].max()
        ),
        "trajectory_file": trajectory_path.relative_to(
            PROJECT_ROOT
        ).as_posix(),
    })

    if (
        row.sample_id % 20 == 0
        or row.sample_id
        == N_TOTAL - 1
    ):
        print(
            "Simulated sample",
            row.sample_id,
        )


simulation_manifest = pd.DataFrame(
    simulation_rows
)

simulation_manifest_path = (
    DATASET_DIR
    / "simulation_manifest.csv"
)

simulation_manifest.to_csv(
    simulation_manifest_path,
    index=False,
)

print()
print(
    simulation_manifest.groupby(
        "split"
    )["samples"].agg(
        [
            "count",
            "min",
            "max",
        ]
    )
)

Simulated sample 0
Simulated sample 20
Simulated sample 40
Simulated sample 60
Simulated sample 80
Simulated sample 100
Simulated sample 119

            count   min   max
split                        
test           18  2501  2501
train          84  2501  2501
validation     18  2501  2501


# 4. Fit the rendering bounds using training trajectories only

To make the image map consistent across the dataset, every sample must use the same physical plotting bounds.

Those bounds are estimated from the **training trajectories only**.

Let

$$
x_{\min}^{\mathrm{train}},
\quad
x_{\max}^{\mathrm{train}},
\quad
z_{\min}^{\mathrm{train}},
\quad
z_{\max}^{\mathrm{train}}
$$

be the extrema over all training states.

A 5% padding margin is added and the resulting bounds are frozen before validation and test images are rendered.

In [4]:
train_manifest = (
    simulation_manifest[
        simulation_manifest[
            "split"
        ] == "train"
    ]
)

x_min_raw = float(
    train_manifest[
        "x_min"
    ].min()
)

x_max_raw = float(
    train_manifest[
        "x_max"
    ].max()
)

z_min_raw = float(
    train_manifest[
        "z_min"
    ].min()
)

z_max_raw = float(
    train_manifest[
        "z_max"
    ].max()
)

x_span = (
    x_max_raw
    - x_min_raw
)

z_span = (
    z_max_raw
    - z_min_raw
)

x_bounds = (
    x_min_raw
    - 0.05 * x_span,
    x_max_raw
    + 0.05 * x_span,
)

z_bounds = (
    z_min_raw
    - 0.05 * z_span,
    z_max_raw
    + 0.05 * z_span,
)

render_config = {
    "projection": "x-z",
    "width": W,
    "height": H,
    "line_width": LINE_WIDTH,
    "x_bounds": list(
        map(float, x_bounds)
    ),
    "z_bounds": list(
        map(float, z_bounds)
    ),
    "bound_fit_split": "train",
    "padding_fraction": 0.05,
}

render_config_path = (
    DATASET_DIR
    / "rendering_config.json"
)

render_config_path.write_text(
    json.dumps(
        render_config,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    json.dumps(
        render_config,
        indent=2,
    )
)

{
  "projection": "x-z",
  "width": 256,
  "height": 256,
  "line_width": 1,
  "x_bounds": [
    -22.649958701538726,
    22.714981412019714
  ],
  "z_bounds": [
    -0.3624469924123681,
    54.8753381465195
  ],
  "bound_fit_split": "train",
  "padding_fraction": 0.05
}


# 5. Render every trajectory using the same map

In [5]:
def physical_to_pixel(
    x_value,
    z_value,
    x_bounds,
    z_bounds,
    width,
    height,
):
    x_min, x_max = x_bounds
    z_min, z_max = z_bounds

    u = (
        (x_value - x_min)
        / (x_max - x_min)
        * (width - 1)
    )

    v = (
        (height - 1)
        - (
            (z_value - z_min)
            / (z_max - z_min)
            * (height - 1)
        )
    )

    return (
        float(u),
        float(v),
    )


def render_trajectory(
    states,
):
    image = Image.new(
        "RGB",
        (W, H),
        (255, 255, 255),
    )

    draw = ImageDraw.Draw(
        image
    )

    points = [
        physical_to_pixel(
            x_value,
            z_value,
            x_bounds,
            z_bounds,
            W,
            H,
        )
        for x_value, z_value
        in zip(
            states[:, 0],
            states[:, 2],
        )
    ]

    draw.line(
        points,
        fill=(0, 0, 0),
        width=LINE_WIDTH,
        joint="curve",
    )

    return image


image_rows = []

for row in conditions.itertuples():
    trajectory_file = (
        TRAJECTORY_DIR
        / f"sample_{row.sample_id:03d}.npz"
    )

    stored = np.load(
        trajectory_file
    )

    states = stored[
        "states"
    ]

    outside = (
        (states[:, 0] < x_bounds[0])
        | (states[:, 0] > x_bounds[1])
        | (states[:, 2] < z_bounds[0])
        | (states[:, 2] > z_bounds[1])
    )

    image = render_trajectory(
        states
    )

    image_path = (
        IMAGE_DIR
        / f"sample_{row.sample_id:03d}.png"
    )

    image.save(
        image_path
    )

    if image.size != (
        W,
        H,
    ):
        raise ValueError(
            "Unexpected image size."
        )

    image_rows.append({
        "sample_id": int(
            row.sample_id
        ),
        "split": row.split,
        "image_file": image_path.relative_to(
            PROJECT_ROOT
        ).as_posix(),
        "width": W,
        "height": H,
        "channels": 3,
        "points_outside_training_render_bounds": int(
            outside.sum()
        ),
        "fraction_outside_training_render_bounds": float(
            outside.mean()
        ),
    })


image_manifest = pd.DataFrame(
    image_rows
)

image_manifest_path = (
    DATASET_DIR
    / "image_manifest.csv"
)

image_manifest.to_csv(
    image_manifest_path,
    index=False,
)

print(
    image_manifest.groupby(
        "split"
    )[
        "fraction_outside_training_render_bounds"
    ].agg(
        [
            "count",
            "mean",
            "max",
        ]
    )
)

            count  mean  max
split                       
test           18   0.0  0.0
train          84   0.0  0.0
validation     18   0.0  0.0


# 6. Create the final dataset manifest

The manifest links each condition vector to its numerical trajectory and rendered image.

In [6]:
dataset_manifest = (
    conditions
    .merge(
        simulation_manifest[
            [
                "sample_id",
                "trajectory_file",
                "samples",
            ]
        ],
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        image_manifest[
            [
                "sample_id",
                "image_file",
                "width",
                "height",
                "channels",
                "fraction_outside_training_render_bounds",
            ]
        ],
        on="sample_id",
        how="left",
        validate="one_to_one",
    )
)

dataset_manifest[
    "dt"
] = DT

dataset_manifest[
    "trajectory_vector_dimension"
] = (
    3
    * dataset_manifest[
        "samples"
    ]
)

dataset_manifest[
    "image_vector_dimension"
] = (
    dataset_manifest[
        "width"
    ]
    * dataset_manifest[
        "height"
    ]
    * dataset_manifest[
        "channels"
    ]
)

manifest_path = (
    DATASET_DIR
    / "dataset_manifest.csv"
)

dataset_manifest.to_csv(
    manifest_path,
    index=False,
)

print(
    dataset_manifest.head().to_string(
        index=False
    )
)

print()
print("Saved:", manifest_path.resolve())

 sample_id     sigma       rho     beta        x0        y0        z0    T split                                                  trajectory_file  samples                                                 image_file  width  height  channels  fraction_outside_training_render_bounds   dt  trajectory_vector_dimension  image_vector_dimension
         0  8.534315 24.161536 2.379947 13.316783 10.676565 32.206152 25.0 train data/hw7_1/part4_conditioned_dataset/trajectories/sample_000.npz     2501 data/hw7_1/part4_conditioned_dataset/images/sample_000.png    256     256         3                                      0.0 0.01                         7503                  196608
         1  9.378573 31.599074 2.469527 -9.781024 -4.023229 13.656476 25.0 train data/hw7_1/part4_conditioned_dataset/trajectories/sample_001.npz     2501 data/hw7_1/part4_conditioned_dataset/images/sample_001.png    256     256         3                                      0.0 0.01                         7503           

# 7. Dataset verification

Every sample should satisfy:

- one seven-dimensional condition vector;
- one trajectory with 2501 states in $$\mathbb{R}^3$$;
- one 256-by-256 RGB image;
- one manifest row linking the three representations.

In [7]:
verification_rows = []

for row in dataset_manifest.itertuples():
    trajectory_path = (
        PROJECT_ROOT
        / row.trajectory_file
    )

    image_path = (
        PROJECT_ROOT
        / row.image_file
    )

    stored = np.load(
        trajectory_path
    )

    states = stored["states"]
    condition = stored[
        "condition"
    ]

    image = Image.open(
        image_path
    ).convert("RGB")

    valid = (
        condition.shape == (7,)
        and states.shape
        == (2501, 3)
        and image.size
        == (W, H)
    )

    verification_rows.append({
        "sample_id": int(
            row.sample_id
        ),
        "split": row.split,
        "condition_shape_ok": (
            condition.shape
            == (7,)
        ),
        "trajectory_shape_ok": (
            states.shape
            == (2501, 3)
        ),
        "image_shape_ok": (
            image.size
            == (W, H)
        ),
        "all_checks_pass": bool(
            valid
        ),
    })


verification = pd.DataFrame(
    verification_rows
)

verification_path = (
    DATASET_DIR
    / "dataset_verification.csv"
)

verification.to_csv(
    verification_path,
    index=False,
)

if not verification[
    "all_checks_pass"
].all():
    raise RuntimeError(
        "At least one dataset sample failed verification."
    )

print(
    verification.groupby(
        "split"
    )[
        "all_checks_pass"
    ].agg(
        [
            "count",
            "sum",
        ]
    )
)

print()
print(
    "All",
    len(verification),
    "samples passed shape verification."
)

            count  sum
split                 
test           18   18
train          84   84
validation     18   18

All 120 samples passed shape verification.


# 8. Fixed preview of conditioned examples

The preview uses six predetermined sample IDs:

$$
0,\ 1,\ 2,\ 84,\ 102,\ 119.
$$

These are chosen by index rather than by visual quality.

The contact sheet is for human inspection only and does not affect the dataset.

In [8]:
preview_ids = [
    0,
    1,
    2,
    84,
    102,
    119,
]

tile_width = W
tile_height = H + 40

sheet = Image.new(
    "RGB",
    (
        3 * tile_width,
        2 * tile_height,
    ),
    (240, 240, 240),
)

draw = ImageDraw.Draw(
    sheet
)

for slot, sample_id in enumerate(
    preview_ids
):
    row = dataset_manifest[
        dataset_manifest[
            "sample_id"
        ] == sample_id
    ].iloc[0]

    image = Image.open(
        PROJECT_ROOT
        / row[
            "image_file"
        ]
    ).convert("RGB")

    grid_row = slot // 3
    grid_col = slot % 3

    left = (
        grid_col
        * tile_width
    )

    top = (
        grid_row
        * tile_height
    )

    sheet.paste(
        image,
        (left, top),
    )

    label = (
        f"id={sample_id}, "
        f"{row['split']}, "
        f"s={row['sigma']:.2f}, "
        f"r={row['rho']:.2f}"
    )

    draw.text(
        (
            left + 5,
            top + H + 10,
        ),
        label,
        fill=(0, 0, 0),
    )


preview_path = (
    RESULT_DIR
    / "part4_conditioned_dataset_preview.png"
)

sheet.save(
    preview_path
)

print("Saved:", preview_path.resolve())

Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.1\result images\hw7_1\part4_conditioned_dataset_preview.png


# 9. Save compact dataset summary statistics

In [9]:
summary = {
    "seed": SEED,
    "total_samples": N_TOTAL,
    "train_samples": N_TRAIN,
    "validation_samples": N_VALIDATION,
    "test_samples": N_TEST,
    "trajectory_samples_per_condition": 2501,
    "trajectory_vector_dimension": 7503,
    "image_height": H,
    "image_width": W,
    "image_channels": 3,
    "image_vector_dimension": 196608,
    "render_bounds_fit_from": "train trajectories only",
    "mean_validation_fraction_outside_render_bounds": float(
        image_manifest.loc[
            image_manifest[
                "split"
            ] == "validation",
            "fraction_outside_training_render_bounds",
        ].mean()
    ),
    "mean_test_fraction_outside_render_bounds": float(
        image_manifest.loc[
            image_manifest[
                "split"
            ] == "test",
            "fraction_outside_training_render_bounds",
        ].mean()
    ),
    "max_validation_fraction_outside_render_bounds": float(
        image_manifest.loc[
            image_manifest[
                "split"
            ] == "validation",
            "fraction_outside_training_render_bounds",
        ].max()
    ),
    "max_test_fraction_outside_render_bounds": float(
        image_manifest.loc[
            image_manifest[
                "split"
            ] == "test",
            "fraction_outside_training_render_bounds",
        ].max()
    ),
}

summary_path = (
    DATASET_DIR
    / "dataset_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2))
print("Saved:", summary_path.resolve())

{
  "seed": 63871,
  "total_samples": 120,
  "train_samples": 84,
  "validation_samples": 18,
  "test_samples": 18,
  "trajectory_samples_per_condition": 2501,
  "trajectory_vector_dimension": 7503,
  "image_height": 256,
  "image_width": 256,
  "image_channels": 3,
  "image_vector_dimension": 196608,
  "render_bounds_fit_from": "train trajectories only",
  "mean_validation_fraction_outside_render_bounds": 0.0,
  "mean_test_fraction_outside_render_bounds": 0.0,
  "max_validation_fraction_outside_render_bounds": 0.0,
  "max_test_fraction_outside_render_bounds": 0.0
}
Saved: C:\Users\benja\OneDrive\Univeristy of Alabama courses\Math 638\Project 2\HW7.1\data\hw7_1\part4_conditioned_dataset\dataset_summary.json


# Part IV Conclusion

The conditioned dataset now explicitly stores corresponding triples

$$
\boxed{
(c,\Gamma_c,I_c).
}
$$

These are paired dataset representations, not mutually invertible maps. The scientific generative direction remains

$$
c\longmapsto\Gamma_c\longmapsto I_c.
$$

The condition vector contains physical/model parameters and the initial state.

The trajectory is generated by integrating the Lorenz equations.

The image is generated only from the numerical trajectory through a frozen rendering map.

This separation will be the central evidence for the Part V reflection.